In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script: Batch Quality Analysis and Production Summary Test Suite
# Purpose: Comprehensive test code for batch quality analysis and production summary in Databricks
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script validates batch quality status, product-wise batch analysis, and batch-wise production summary using PySpark DataFrame API. It includes schema validation, data type checks, null handling, duplicate detection, edge case testing, and output structure assertions. All tests are performed on the Unity Catalog table 'purgo_databricks.purgo_playground.batch_qc'.

# Required imports for PySpark DataFrame operations and testing
from pyspark.sql import DataFrame  
from pyspark.sql import functions as F  
from pyspark.sql.types import (  
    StructType, StructField, StringType, DoubleType, DateType, LongType
)
from pyspark.sql.utils import AnalysisException  

# -- Test Setup: Define expected schema for batch_qc table
expected_schema = StructType([
    StructField("batch_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("production_date", DateType(), True),
    StructField("quantity", LongType(), True),
    StructField("quality_check_score", DoubleType(), True)
])

# -- Test Setup: Read source table with error handling for missing/invalid data
def read_batch_qc_table():
    """
    Reads the batch_qc table from Unity Catalog with error handling.
    Returns:
        DataFrame: The batch_qc DataFrame.
    Raises:
        Exception: If the table is missing or schema is invalid.
    """
    try:
        df = spark.read.table("purgo_databricks.purgo_playground.batch_qc")
        # Schema validation: Ensure column names and types match expected
        actual_fields = [(f.name, f.dataType.typeName()) for f in df.schema.fields]
        expected_fields = [(f.name, f.dataType.typeName()) for f in expected_schema.fields]
        assert actual_fields == expected_fields, f"Schema mismatch: {actual_fields} != {expected_fields}"
        # Column count validation
        assert len(df.columns) == len(expected_schema.fields), "Column count mismatch"
        return df
    except AnalysisException as e:
        raise Exception(f"Table not found: {e}")
    except AssertionError as e:
        raise Exception(f"Schema validation failed: {e}")

# -- Test: Validate required fields are present and not null
def validate_required_fields(df: DataFrame):
    """
    Validates that required fields are present and not null.
    Args:
        df (DataFrame): Input DataFrame.
    Raises:
        Exception: If required fields are missing or null.
    """
    required_fields = ["batch_id", "product_name", "quality_check_score"]
    for field in required_fields:
        if field not in df.columns:
            raise Exception(f"Missing required field: {field}")
        null_count = df.filter(F.col(field).isNull()).count()
        if null_count > 0:
            raise Exception(f"Missing required field: {field} (null values detected)")

# -- Test: Validate data types of fields
def validate_field_types(df: DataFrame):
    """
    Validates data types of critical fields.
    Args:
        df (DataFrame): Input DataFrame.
    Raises:
        Exception: If type mismatch is detected.
    """
    # batch_id and product_name must be string
    string_fields = ["batch_id", "product_name"]
    for field in string_fields:
        if not isinstance(df.schema[field].dataType, StringType):
            raise Exception(f"Type mismatch in field: {field} (expected string, got {df.schema[field].dataType})")
    # quality_check_score must be double
    if not isinstance(df.schema["quality_check_score"].dataType, DoubleType):
        raise Exception(f"Type mismatch in field: quality_check_score (expected double, got {df.schema['quality_check_score'].dataType})")
    # quantity must be long
    if not isinstance(df.schema["quantity"].dataType, LongType):
        raise Exception(f"Type mismatch in field: quantity (expected long, got {df.schema['quantity'].dataType})")

# -- Test: Detect duplicate batch_id
def detect_duplicate_batch_id(df: DataFrame):
    """
    Detects duplicate batch_id values.
    Args:
        df (DataFrame): Input DataFrame.
    Raises:
        Exception: If duplicate batch_id is found.
    """
    dup_df = df.groupBy("batch_id").count().filter(F.col("count") > 1)
    dup_ids = [row["batch_id"] for row in dup_df.collect()]
    if dup_ids:
        raise Exception(f"Duplicate batch_id detected: {', '.join(dup_ids)}")

# -- Test: Validate quality_check_score values (null, negative, NaN)
def validate_quality_check_score(df: DataFrame):
    """
    Validates quality_check_score values for null, negative, or NaN.
    Args:
        df (DataFrame): Input DataFrame.
    Raises:
        Exception: If invalid quality_check_score is found.
    """
    invalid_score_df = df.filter(
        (F.col("quality_check_score").isNull()) |
        (F.col("quality_check_score") < 0) |
        (F.isnan(F.col("quality_check_score")))
    )
    invalid_rows = invalid_score_df.select("batch_id", "quality_check_score").collect()
    for row in invalid_rows:
        raise Exception(f"Invalid quality_check_score: {row['quality_check_score']} in batch_id {row['batch_id']}")

# -- Test: Null or empty product_name handling
def validate_product_name(df: DataFrame):
    """
    Excludes batches with null or empty product_name from product-wise aggregation.
    Args:
        df (DataFrame): Input DataFrame.
    Returns:
        DataFrame: Filtered DataFrame.
    """
    null_or_empty_df = df.filter(
        (F.col("product_name").isNull()) | (F.trim(F.col("product_name")) == "")
    )
    for row in null_or_empty_df.select("batch_id").collect():
        print(f"Error: Null or empty product_name in batch_id {row['batch_id']}")
    return df.filter(
        (F.col("product_name").isNotNull()) & (F.trim(F.col("product_name")) != "")
    )

# -- Transformation: Compute quality_status column
def compute_quality_status(df: DataFrame):
    """
    Computes quality_status column based on quality_check_score.
    Args:
        df (DataFrame): Input DataFrame.
    Returns:
        DataFrame: DataFrame with quality_status column.
    """
    return df.withColumn(
        "quality_status",
        F.when(F.col("quality_check_score") < 97, F.lit("fail")).otherwise(F.lit("pass"))
    )

# -- Test: Edge case - boundary quality_check_score
def test_quality_status_boundaries(df: DataFrame):
    """
    Tests quality_status computation for boundary values.
    Args:
        df (DataFrame): Input DataFrame.
    Raises:
        AssertionError: If boundary logic fails.
    """
    boundary_cases = [
        (97.0, "pass"),
        (96.99, "fail"),
        (100.0, "pass"),
        (0.0, "fail")
    ]
    for score, expected_status in boundary_cases:
        test_df = df.filter(F.col("quality_check_score") == score)
        if test_df.count() > 0:
            actual_status = test_df.select("quality_status").first()["quality_status"]
            assert actual_status == expected_status, f"Boundary test failed for score {score}: {actual_status} != {expected_status}"

# -- Aggregation: Product-wise batch analysis
def product_wise_analysis(df: DataFrame):
    """
    Aggregates product-wise batch analysis.
    Args:
        df (DataFrame): Input DataFrame with quality_status.
    Returns:
        DataFrame: Product-wise analysis DataFrame.
    """
    grouped = df.groupBy("product_name").agg(
        F.count("batch_id").alias("total_batches"),
        F.sum(F.when(F.col("quality_status") == "pass", 1).otherwise(0)).alias("passed_batches"),
        F.sum(F.when(F.col("quality_status") == "fail", 1).otherwise(0)).alias("failed_batches")
    )
    result = grouped.withColumn(
        "percentage_passed",
        F.round(F.col("passed_batches") / F.col("total_batches") * 100, 2)
    )
    return result.select(
        "product_name", "total_batches", "passed_batches", "failed_batches", "percentage_passed"
    )

# -- Aggregation: Batch-wise quality status
def batch_wise_quality_status(df: DataFrame):
    """
    Selects batch-wise quality status columns.
    Args:
        df (DataFrame): Input DataFrame with quality_status.
    Returns:
        DataFrame: Batch-wise quality status DataFrame.
    """
    return df.select("batch_id", "product_name", "quality_check_score", "quality_status")

# -- Aggregation: Batch-wise summary of production
def batch_wise_summary(df: DataFrame):
    """
    Aggregates batch-wise production summary.
    Args:
        df (DataFrame): Input DataFrame with quality_status.
    Returns:
        DataFrame: Batch-wise summary DataFrame.
    """
    agg = df.agg(
        F.count("batch_id").alias("total_batches"),
        F.sum(F.when(F.col("quality_status") == "pass", 1).otherwise(0)).alias("total_passed_batches"),
        F.sum(F.when(F.col("quality_status") == "fail", 1).otherwise(0)).alias("total_failed_batches")
    )
    return agg.select("total_batches", "total_passed_batches", "total_failed_batches")

# -- Test: Output structure validation
def validate_output_structure(product_df: DataFrame, batch_status_df: DataFrame, summary_df: DataFrame):
    """
    Validates output DataFrame structures.
    Args:
        product_df (DataFrame): Product-wise analysis DataFrame.
        batch_status_df (DataFrame): Batch-wise quality status DataFrame.
        summary_df (DataFrame): Batch-wise summary DataFrame.
    Raises:
        AssertionError: If output structure is invalid.
    """
    assert product_df.columns == [
        "product_name", "total_batches", "passed_batches", "failed_batches", "percentage_passed"
    ], f"Product-wise analysis columns mismatch: {product_df.columns}"
    assert batch_status_df.columns == [
        "batch_id", "product_name", "quality_check_score", "quality_status"
    ], f"Batch-wise quality status columns mismatch: {batch_status_df.columns}"
    assert summary_df.columns == [
        "total_batches", "total_passed_batches", "total_failed_batches"
    ], f"Batch-wise summary columns mismatch: {summary_df.columns}"

# -- Test: Consistency of output counts
def validate_output_counts(product_df: DataFrame, batch_status_df: DataFrame, summary_df: DataFrame):
    """
    Validates consistency of total_batches across outputs.
    Args:
        product_df (DataFrame): Product-wise analysis DataFrame.
        batch_status_df (DataFrame): Batch-wise quality status DataFrame.
        summary_df (DataFrame): Batch-wise summary DataFrame.
    Raises:
        AssertionError: If counts are inconsistent.
    """
    total_batches_summary = summary_df.select("total_batches").first()["total_batches"]
    total_batches_product = product_df.select(F.sum("total_batches")).first()[0]
    total_batches_status = batch_status_df.count()
    assert total_batches_summary == total_batches_product, f"Total batches mismatch: summary={total_batches_summary}, product={total_batches_product}"
    assert total_batches_summary == total_batches_status, f"Total batches mismatch: summary={total_batches_summary}, status={total_batches_status}"

# -- Main Test Execution
def run_all_tests():
    """
    Runs all test cases for batch quality analysis and production summary.
    """
    # Read source table
    batch_qc_df = read_batch_qc_table()
    # Validate required fields
    validate_required_fields(batch_qc_df)
    # Validate field types
    validate_field_types(batch_qc_df)
    # Detect duplicate batch_id
    detect_duplicate_batch_id(batch_qc_df)
    # Validate quality_check_score values
    validate_quality_check_score(batch_qc_df)
    # Exclude null/empty product_name for product-wise analysis
    batch_qc_df_valid = validate_product_name(batch_qc_df)
    # Compute quality_status
    batch_qc_status_df = compute_quality_status(batch_qc_df)
    # Test boundary cases for quality_status
    test_quality_status_boundaries(batch_qc_status_df)
    # Product-wise analysis
    product_wise_analysis_df = product_wise_analysis(compute_quality_status(batch_qc_df_valid))
    # Batch-wise quality status
    batch_wise_quality_status_df = batch_wise_quality_status(batch_qc_status_df)
    # Batch-wise summary
    batch_wise_summary_df = batch_wise_summary(batch_qc_status_df)
    # Validate output structure
    validate_output_structure(product_wise_analysis_df, batch_wise_quality_status_df, batch_wise_summary_df)
    # Validate output counts consistency
    validate_output_counts(product_wise_analysis_df, batch_wise_quality_status_df, batch_wise_summary_df)
    # Print results for manual inspection (optional)
    print("Product-wise Analysis:")
    product_wise_analysis_df.show()
    print("Batch-wise Quality Status:")
    batch_wise_quality_status_df.show()
    print("Batch-wise Summary:")
    batch_wise_summary_df.show()

# -- Run all tests
run_all_tests()

# -- Delta Lake Operations: Test MERGE, UPDATE, DELETE (if applicable)
# Note: These operations are not required for the main analysis, but included for completeness.

def test_delta_lake_operations():
    """
    Tests Delta Lake MERGE, UPDATE, DELETE operations on a test table.
    """
    test_table = "purgo_databricks.purgo_playground.batch_qc_test_delta"
    # Create test Delta table
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {test_table} (
            batch_id STRING NOT NULL,
            product_name STRING,
            quality_check_score DOUBLE,
            quality_status STRING,
            CONSTRAINT quality_status_values CHECK (quality_status IN ("pass", "fail"))
        ) USING DELTA
    """)
    # Insert test data
    data = [
        ("BATCH010", "WidgetX", 99.0, "pass"),
        ("BATCH011", "WidgetY", 95.0, "fail")
    ]
    delta_df = spark.createDataFrame(data, ["batch_id", "product_name", "quality_check_score", "quality_status"])
    delta_df.write.format("delta").mode("overwrite").saveAsTable(test_table)
    # MERGE: Upsert a new batch
    spark.sql(f"""
        MERGE INTO {test_table} AS target
        USING (SELECT "BATCH011" AS batch_id, "WidgetY" AS product_name, 97.5 AS quality_check_score, "pass" AS quality_status) AS source
        ON target.batch_id = source.batch_id
        WHEN MATCHED THEN UPDATE SET target.quality_check_score = source.quality_check_score, target.quality_status = source.quality_status
        WHEN NOT MATCHED THEN INSERT (batch_id, product_name, quality_check_score, quality_status) VALUES (source.batch_id, source.product_name, source.quality_check_score, source.quality_status)
    """)
    # UPDATE: Change quality_status
    spark.sql(f"""
        UPDATE {test_table}
        SET quality_status = "fail"
        WHERE batch_id = "BATCH010"
    """)
    # DELETE: Remove a batch
    spark.sql(f"""
        DELETE FROM {test_table}
        WHERE batch_id = "BATCH011"
    """)
    # Cleanup: Drop test table
    spark.sql(f"DROP TABLE IF EXISTS {test_table}")

# -- Run Delta Lake operations test
test_delta_lake_operations()

# -- Performance Test: Validate aggregation performance for large datasets
def performance_test_large_aggregation():
    """
    Performance test for product-wise aggregation on large synthetic dataset.
    """
    # Generate synthetic data
    num_records = 100000
    import random  
    from datetime import date, timedelta  
    products = ["WidgetA", "WidgetB", "WidgetC", "WidgetD", "WidgetE"]
    start_date = date(2025, 10, 1)
    data = []
    for i in range(num_records):
        batch_id = f"BATCH{1000+i}"
        product_name = random.choice(products)
        production_date = start_date + timedelta(days=random.randint(0, 30))
        quantity = random.randint(10, 500)
        score = round(random.uniform(90, 100), 2)
        data.append((batch_id, product_name, production_date, quantity, score))
    large_df = spark.createDataFrame(data, schema=expected_schema)
    # Compute quality_status
    large_df_status = compute_quality_status(large_df)
    # Time aggregation
    import time  
    start = time.time()
    product_wise_large = product_wise_analysis(large_df_status)
    duration = time.time() - start
    assert duration < 10, f"Aggregation took too long: {duration} seconds"
    print(f"Performance test passed: Aggregation completed in {duration:.2f} seconds")

# -- Run performance test
performance_test_large_aggregation()

# -- Window Function Test: Calculate rolling pass rate per product
def test_window_functions(df: DataFrame):
    """
    Tests window function for rolling pass rate per product.
    Args:
        df (DataFrame): Input DataFrame with quality_status.
    """
    from pyspark.sql.window import Window  
    window_spec = Window.partitionBy("product_name").orderBy("production_date").rowsBetween(-2, 0)
    rolling_pass_rate_df = df.withColumn(
        "rolling_pass_rate",
        F.round(
            F.sum(F.when(F.col("quality_status") == "pass", 1).otherwise(0)).over(window_spec) /
            F.count("batch_id").over(window_spec) * 100, 2
        )
    )
    # Validate rolling_pass_rate is between 0 and 100
    invalid_rate_count = rolling_pass_rate_df.filter(
        (F.col("rolling_pass_rate") < 0) | (F.col("rolling_pass_rate") > 100)
    ).count()
    assert invalid_rate_count == 0, "Invalid rolling_pass_rate detected"
    print("Window function test passed: rolling_pass_rate within valid range")

# -- Run window function test
batch_qc_df = read_batch_qc_table()
batch_qc_status_df = compute_quality_status(batch_qc_df)
test_window_functions(batch_qc_status_df)

# -- Cleanup operations: No temp views or temp tables used, so no cleanup required

# -- End of test script
